# Filaments: training on how many annotators drew each pixel

Full-disk H-alpha images of the Sun, 2048 square. Outline every dark filament
**individually**, and no two masks in a frame may share a pixel. Scored by
Panoptic Quality: a match needs IoU above 0.5, and a prediction that falls
short costs more than silence. A U-Net marks filament pixels; instances are
the connected regions of the thresholded map. This notebook does the lot --
train, score, tune the post-processing, predict the test frames, write a
submission -- and holds no model code of its own, importing it from the
repository the first cell clones.

**What is being tested.** 707 training frames carry 1,154 sets of outlines:
296 of them were annotated by two or three people, who differ by 2.6 filaments
each. The metric scores such a frame once per annotator, so one prediction
faces all of them -- but training feeds it each tracing in turn and leaves it
to average them itself. What comes out is not a vote share but an uncalibrated
compromise, and every threshold from 0.3 to 0.7 scores within 0.001 of the
rest, so the threshold is not a parameter at all.

This run asks for the share instead -- 0, 1/3, 2/3 or 1 -- leaving the samples
and their weighting unchanged. Cross-entropy teaches it. Dice would prefer
predicting 1.0 on a target of 2/3, so it reads a majority vote instead and goes
on handling the imbalance.

That makes the decision answerable. A filament **k of n** people drew, emitted
at IoU **u**, adds `k*u` to the numerator and `0.5*n` to the denominator
against staying silent, so it pays when `k*u > 0.5*n*PQ`. At PQ 0.40, one of
three needs IoU 0.6; two of three needs 0.3. A calibrated map turns that into
a threshold.

**The number to beat is PQ 0.3756** on the 142 validation frames of a frozen
fold, where the previous run left it. The comparison is made at that run's
post-processing settings before anything is retuned, so what moves is the model
rather than the tuning.

Before running: **GPU T4 x2**, **Internet on**, competition data attached as an
input. Save with **Save Version** and set **Save output** under *Advanced
Settings* -- a Quick Save discards the checkpoint and the probability maps.
About 75 minutes, and everything after training reads the saved maps, so it
needs no GPU.

## 1. Clone the repository

Cloned rather than pip-installed: `configs/paths.yaml` and the frozen splits in
`configs/splits/` sit beside the package rather than inside it, and a wheel
would leave them behind -- the run would then be validated on a different set
of frames than every other run.

To reproduce this exact run later, put its commit hash in `REF`.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase4-vote-targets"  # or a commit hash, for a run to reproduce later
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; its CUDA build of torch stays as it is.
!pip install -q segmentation-models-pytorch

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import logging

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import filament

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print("filament", filament.__version__)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

`MAGFILO_ROOT` overrides the dataset root in `configs/paths.yaml`. The
annotation file is searched for rather than hard-coded, because the input
directory is named after whatever the competition data was attached as.

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

# <root>/train/<annotations> -> <root>
os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))
print("test images: ", len(list(paths.test_images.glob("*.jpeg"))))

WORK = Path("/kaggle/working")

## 3. Train

A U-Net with a ResNet-34 encoder, at 1024 pixels on two channels: the frame and
a locally contrast-equalised copy. Fifteen epochs rather than forty, because
the longer schedule's validation loss bottomed out around epoch 8 and the
shorter one also scored better, PQ 0.3649 against 0.3454.

`num_workers` is raised from the committed value, which is 0 for Windows where
every worker would re-read the 48 MB annotation file.

In [ ]:
from dataclasses import replace

from filament.training.config import TrainConfig
from filament.training.loop import train

config = replace(
    TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase4_votes.yaml"),
    num_workers=2,
    output_dir=WORK / "phase4_votes",
)
print(config)
assert config.vote_targets, "This notebook is the vote-share run; the config says otherwise."

In [ ]:
result = train(config)
print("best epoch", result.best_epoch, "validation loss", round(result.best_val_loss, 4))
print("checkpoint", result.checkpoint)

In [ ]:
epochs = [item.epoch for item in result.history]
figure, axes = plt.subplots(figsize=(7, 3.2))
axes.plot(
    epochs,
    [item.train_loss for item in result.history],
    label="train",
    linewidth=2,
    color="#2a78d6",
)
axes.plot(
    epochs,
    [item.val_loss for item in result.history],
    label="validation",
    linewidth=2,
    color="#eb6834",
)
axes.set_xlabel("epoch")
axes.set_ylabel("Dice + cross-entropy")
axes.set_title("Training and validation loss")
axes.legend(frameon=False)
axes.grid(axis="y", linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"training time: {sum(item.seconds for item in result.history) / 60:.1f} min")

## 4. Write the validation probability maps

Everything below reads these files rather than the model: thresholding, joining
fragments, filtering by area, scoring, and the sweep. Writing them once is what
keeps the rest off the GPU, and a byte per pixel resolves the probability far
finer than any threshold in use.

The solar disk is found here too. `extract_instances` needs it in the
coordinates of the map, and detecting it costs 20 ms a frame -- worth doing
once rather than inside every sweep point.

In [ ]:
from filament.data.coco import load_annotations
from filament.data.disk import detect_disk
from filament.data.image import load_grayscale
from filament.data.split import load_fold
from filament.evaluation import write_probability_maps
from filament.postprocess.search import load_maps
from filament.training.loop import load_checkpoint

model, stored = load_checkpoint(result.checkpoint)
dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(config.fold, f"{CHECKOUT}/configs/splits").val
print(f"fold {config.fold}: {len(val_stems)} validation frames")

VAL_MAPS = WORK / "prob_fold0_votes"
write_probability_maps(
    model,
    [paths.train_images / f"{stem}.jpeg" for stem in val_stems],
    VAL_MAPS,
    size=config.image_size,
    device="cuda",
)
val_maps = load_maps(VAL_MAPS)

val_disks = {}
for stem in val_stems:
    frame = load_grayscale(paths.train_images / f"{stem}.jpeg")
    val_disks[stem] = detect_disk(frame).scaled(config.image_size / frame.shape[0])

size_mb = sum(path.stat().st_size for path in VAL_MAPS.glob("*.npy")) / 1e6
print(f"{len(val_maps)} maps, {size_mb:.0f} MB")

## 5. Score it against the previous run

The comparison that decides whether this change worked. Same 142 frames, same
evaluator, and the same post-processing that produced **PQ 0.3756** -- nothing
retuned yet, so what moves is the model.

For reference on how far there is to go: taking one person's outlines and
submitting them as the prediction for that frame scores PQ 0.7295 here, and
0.589 counting only the frames more than one person annotated. That is roughly
what annotating as well as a human is worth.

Encoding the ground truth costs about as long as everything else here and never
changes, so it is built once and reused by every sweep point below.

In [ ]:
from filament.metrics.pq import compute_pq
from filament.postprocess.search import Setting, predict_from_maps
from filament.submit.rle import masks_to_gt_df

BASELINE = {"pq": 0.3756, "sq": 0.6555, "rq": 0.5730, "tp": 968, "fp": 616, "fn": 827}
FROZEN = Setting(
    {"threshold": 0.5, "min_area": 400, "join_gap": 24.0, "join_angle": 50.0, "join_offset": 20.0}
)

gt_df = masks_to_gt_df(dataset, val_stems)
print(f"{len(gt_df)} annotated filaments over {len(val_stems)} frames")

symmetric = compute_pq(gt_df, predict_from_maps(val_maps, FROZEN, val_disks))
print(f"vote shares, previous settings: {symmetric}")
print(
    f"previous model, same settings: PQ={BASELINE['pq']:.4f} SQ={BASELINE['sq']:.4f} "
    f"RQ={BASELINE['rq']:.4f} TP={BASELINE['tp']} FP={BASELINE['fp']} FN={BASELINE['fn']}"
)
print(
    f"difference: {symmetric.pq - BASELINE['pq']:+.4f} PQ, "
    f"{symmetric.sq - BASELINE['sq']:+.4f} SQ, {symmetric.rq - BASELINE['rq']:+.4f} RQ"
)

## 6. Did the map become a vote share?

The point of the change is that the output should rise with how many annotators
drew a pixel, because that is what makes the threshold a decision rather than an
arbitrary cut. It is not guaranteed: cross-entropy pulls the output towards the
share while Dice pulls it towards one, and on a target of two thirds the two
settle around 0.75.

Two numbers answer it, and the distance from the diagonal is not one of them:
the background fills the bottom bin with 99.6% of every frame and drags any
pixel-weighted average to nothing.

- **middle_share** -- how much of the marked area the model put somewhere other
  than the two extremes. Previously **0.207**.
- **middle_span** -- how far the observed share of annotators moves across that
  middle. Previously **0.139**, from 0.30 to 0.44, so a threshold placed
  anywhere in there was choosing between pixels people had agreed about
  equally.

Either one staying where it was means this run did not do what it set out to
do, whatever the PQ says.

In [ ]:
from filament.metrics.calibration import calibration

BASELINE_CALIBRATION = {"middle_share": 0.207, "middle_span": 0.139}

curve = calibration(val_maps, dataset.by_stem())
calibration_table = pd.DataFrame(curve.to_rows())
print(f"vote shares:   {curve}")
print(
    f"previous model: middle_share={BASELINE_CALIBRATION['middle_share']:.3f} "
    f"middle_span={BASELINE_CALIBRATION['middle_span']:.3f}"
)
calibration_table

In [ ]:
populated = curve.populated
figure, axes = plt.subplots(figsize=(4.6, 4.4))
axes.plot([0, 1], [0, 1], linewidth=1, color="#b8b5ad", linestyle="--", label="perfect")
axes.plot(
    [item.predicted for item in populated],
    [item.observed for item in populated],
    marker="o",
    linewidth=2,
    color="#2a78d6",
    label="measured",
)
axes.set_xlabel("probability the model gave")
axes.set_ylabel("share of annotators who drew it")
axes.set_title("Is the output a vote share?")
axes.set_xlim(0, 1)
axes.set_ylim(0, 1)
axes.legend(frameon=False)
axes.grid(linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(WORK / "calibration.png", dpi=140)
plt.show()

## 7. Look at the maps

The calibration curve says whether the middle of the range carries anything.
These panels say what it looks like, which is what catches the failures a
summary number cannot name -- a map that responds to the limb, or one whose
uncertainty sits along every boundary rather than on the filaments people
disagreed about.

Three views, each answering something different:

- **The histogram** shows how the probability is distributed. Previously 99.6%
  of pixels sat below 0.05 and most of the rest above 0.9, with only 0.133% of
  the disk in between. A run that worked should have visibly more weight in
  the middle, and should start to resemble the votes beside it.
- **A whole frame** places the predictions on the disk, against what the
  annotators drew.
- **Close-ups of contested filaments**, where the annotators disagreed. This
  is where a vote share should show itself: the parts only one of three people
  drew should come out dimmer than the parts all three drew, rather than
  equally bright.

In [ ]:
from filament.data.dataset import build_vote_target
from filament.data.image import to_model_input
from filament.postprocess.instances import extract_instances

inside_disk = np.concatenate(
    [
        probability[val_disks[stem].mask(*probability.shape, margin=0.0).astype(bool)]
        for stem, probability in val_maps.items()
    ]
)
shares = np.concatenate(
    [
        build_vote_target(dataset.by_stem()[stem], config.image_size).reshape(-1)
        for stem in list(val_maps)[:40]
    ]
)

figure, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for panel, (values, title, colour) in enumerate(
    (
        (inside_disk, "what the model predicted", "#2a78d6"),
        (shares, "what the annotators voted (40 frames)", "#eb6834"),
    )
):
    axes[panel].hist(values, bins=50, range=(0, 1), color=colour, log=True)
    axes[panel].set_title(title)
    axes[panel].set_xlabel("value")
    axes[panel].set_ylabel("pixels (log)")
    axes[panel].spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(WORK / "value_histogram.png", dpi=140)
plt.show()

middle = float(((inside_disk > 0.05) & (inside_disk < 0.90)).mean())
print(f"{middle:.3%} of the pixels inside the disk sit between 0.05 and 0.90")

In [ ]:
DISPLAY_MIN_AREA = int(FROZEN.values["min_area"] / (2048 / config.image_size) ** 2)


def instance_labels(stem):
    """The instances this map yields at the previous run's settings.

    The same settings section 5 scored, so the panels below show what the
    comparison there was made on. The sweep has not run yet.
    """
    instances = extract_instances(
        val_maps[stem],
        disk=val_disks[stem],
        output_size=config.image_size,
        **{**FROZEN.values, "min_area": DISPLAY_MIN_AREA},
    )
    labels = np.zeros((config.image_size, config.image_size), dtype=np.int32)
    for number, instance in enumerate(instances, start=1):
        labels[instance.mask] = number
    return labels, len(instances)


def frame_at(stem):
    """What the model is shown: the contrast-equalised channel, in [0, 1].

    The raw frame is nearly uniform at this scale -- a filament is a few
    percent darker than the disk around it -- so displaying it says nothing.
    This is the second channel of the model's input, which is the same frame
    after local contrast equalisation.
    """
    grey = load_grayscale(paths.train_images / f"{stem}.jpeg")
    return to_model_input(grey, config.image_size)[1]


# A frame several people annotated, so that a vote share has something to say.
contested = [stem for stem in val_maps if len(dataset.by_stem()[stem]) >= 2]
overview_stem = contested[0] if contested else next(iter(val_maps))
votes = build_vote_target(dataset.by_stem()[overview_stem], config.image_size)
labels, count = instance_labels(overview_stem)

figure, axes = plt.subplots(1, 4, figsize=(15, 4))
panels = [
    (frame_at(overview_stem), "what the model sees", "gray", 1.0),
    (val_maps[overview_stem], "predicted", "magma", 1.0),
    (votes, "vote share", "magma", 1.0),
    (labels, f"instances at the old settings ({count})", "nipy_spectral", max(count, 1)),
]
for axis, (image, title, colour, top) in zip(axes, panels, strict=True):
    axis.imshow(image, cmap=colour, vmin=0, vmax=top, interpolation="nearest")
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])
figure.suptitle(f"{overview_stem} - {len(dataset.by_stem()[overview_stem])} annotators")
plt.tight_layout()
plt.savefig(WORK / "overview.png", dpi=140)
plt.show()

In [ ]:
WINDOW = 200


def contested_windows(stem, limit=2):
    """Boxes around filaments the annotators did not agree on.

    Where a vote share should be visible at all: a part everyone drew and a
    part only some did, close enough to appear in one crop.
    """
    votes = build_vote_target(dataset.by_stem()[stem], config.image_size)
    disputed = ((votes > 0) & (votes < 0.999)).astype(np.uint8)
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(disputed, connectivity=8)
    order = sorted(range(1, count), key=lambda index: -stats[index, cv2.CC_STAT_AREA])
    boxes = []
    for index in order[:limit]:
        row, column = (int(value) for value in centroids[index][::-1])
        half = WINDOW // 2
        row = min(max(row, half), config.image_size - half)
        column = min(max(column, half), config.image_size - half)
        boxes.append((row - half, row + half, column - half, column + half))
    return votes, boxes


picked = []
for stem in contested:
    votes, boxes = contested_windows(stem, limit=1)
    if boxes:
        picked.append((stem, votes, boxes[0]))
    if len(picked) == 3:
        break

if not picked:
    print("No frame in this fold has a filament the annotators disagreed about.")
else:
    figure, axes = plt.subplots(len(picked), 4, figsize=(13, 3.6 * len(picked)), squeeze=False)
    shaded = None
    for row, (stem, votes, (top, bottom, left, right)) in enumerate(picked):
        labels, _ = instance_labels(stem)
        crops = [
            (frame_at(stem)[top:bottom, left:right], "what the model sees", "gray"),
            (val_maps[stem][top:bottom, left:right], "predicted", "magma"),
            (votes[top:bottom, left:right], "vote share", "magma"),
            (labels[top:bottom, left:right] > 0, "instances at the old settings", "gray"),
        ]
        for column, (image, title, colour) in enumerate(crops):
            axis = axes[row][column]
            drawn = axis.imshow(image, cmap=colour, vmin=0, vmax=1, interpolation="nearest")
            if column == 1:
                shaded = drawn
            axis.set_xticks([])
            axis.set_yticks([])
            if row == 0:
                axis.set_title(title)
        axes[row][0].set_ylabel(f"{stem}\n{len(dataset.by_stem()[stem])} annotators", fontsize=8)
    figure.colorbar(shaded, ax=axes[:, 1:3], shrink=0.55, label="value")
    plt.savefig(WORK / "contested.png", dpi=140, bbox_inches="tight")
    plt.show()

## 8. Sweep the post-processing

Two sweeps rather than one grid over everything: the threshold and the minimum
area interact, while the joining tolerances did not move the score at all last
time, so pairing them all would cost hundreds of points to learn the same
thing.

The threshold range is wider than before. Last time it was swept from 0.3 to
0.7 and every value scored within 0.001; if the map now carries a share, the
ends should start to differ.

Each point is scored on all 142 frames, and the value taken is the middle of
the widest near-best run rather than the outright best. On 142 frames the top
point among a few dozen is partly an accident of those frames: last time it
beat the middle of the plateau by 0.0016, which is not a result. A value whose
neighbours are also good is the one that survives being applied elsewhere.

In [ ]:
from filament.postprocess.search import grid, sweep

thresholds = [round(0.1 * step, 2) for step in range(1, 10)]
first = sweep(
    val_maps,
    gt_df,
    grid(threshold=thresholds, min_area=[200, 400, 600], join_gap=[24.0]),
    disks=val_disks,
)
first.table

In [ ]:
threshold, threshold_plateau = first.plateau("threshold")
min_area, area_plateau = first.plateau("min_area")
print(f"threshold: plateau {threshold_plateau} -> {threshold}")
print(f"min_area:  plateau {area_plateau} -> {min_area}")
print(f"best single point: {first.best.setting} -> PQ {first.best.pq.pq:.4f}")

In [ ]:
second = sweep(
    val_maps,
    gt_df,
    grid(
        threshold=[threshold],
        min_area=[min_area],
        join_gap=[0.0, 8.0, 16.0, 24.0, 32.0, 48.0],
    ),
    disks=val_disks,
)
join_gap, gap_plateau = second.plateau("join_gap")
print(f"join_gap: plateau {gap_plateau} -> {join_gap}")
second.table

In [ ]:
CHOSEN = Setting(
    {
        "threshold": float(threshold),
        "min_area": int(min_area),
        "join_gap": float(join_gap),
        "join_angle": 50.0,
        "join_offset": 20.0,
    }
)
tuned = compute_pq(gt_df, predict_from_maps(val_maps, CHOSEN, val_disks))
print(f"chosen:    {CHOSEN}")
print(f"tuned:     {tuned}")
print(f"symmetric: {symmetric}")
print(f"previous: PQ={BASELINE['pq']:.4f}")

## 9. Predict the test set

The overlap check is not optional: Kaggle rejects a submission whose masks share
a pixel, and the rejected attempt still uses one of the five allowed per day.

Two submissions are written. The one to send is `submission.csv`, from the
settings the sweep chose. `submission_frozen.csv` uses the previous run's
settings unchanged, as something to fall back on -- it costs one extra pass
over maps that are already in memory.

In [ ]:
from filament.metrics.overlap import check_no_overlap
from filament.submit.rle import write_submission

test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
TEST_MAPS = WORK / "prob_test_votes"
write_probability_maps(
    model,
    [paths.test_images / f"{stem}.jpeg" for stem in test_stems],
    TEST_MAPS,
    size=config.image_size,
    device="cuda",
)
test_maps = load_maps(TEST_MAPS)

test_disks = {}
for stem in test_stems:
    frame = load_grayscale(paths.test_images / f"{stem}.jpeg")
    test_disks[stem] = detect_disk(frame).scaled(config.image_size / frame.shape[0])
print(f"{len(test_maps)} test maps")

In [ ]:
submissions = {}
for name, setting in (("submission.csv", CHOSEN), ("submission_frozen.csv", FROZEN)):
    frame = predict_from_maps(test_maps, setting, test_disks)
    path = write_submission(frame, WORK / name)
    check_no_overlap(path)
    covered = frame["filament_id"].str.rsplit("_", n=1).str[0].nunique()
    submissions[name] = frame
    print(
        f"{name}: {len(frame)} masks over {covered} of {len(test_maps)} frames, "
        f"{len(test_maps) - covered} with none"
    )

submissions["submission.csv"].head()

## 10. Package the results

The submission, the numbers, the sweep tables and the figures go into one
archive, along with the validation probability maps: the analysis that decides
what to try next -- where the false positives come from, how far the masks fall
short -- runs off those maps and should not need a GPU session each time.

The test maps and the checkpoint stay in the notebook output instead, where a
later run can attach them as an input dataset rather than anyone downloading
and re-uploading them.

Files are streamed into the archive from where they already are; nothing is
copied into a staging directory first.

In [ ]:
import json
import zipfile

summary = {
    "commit": REF,
    "symmetric": {
        "pq": symmetric.pq,
        "sq": symmetric.sq,
        "rq": symmetric.rq,
        "tp": symmetric.tp,
        "fp": symmetric.fp,
        "fn": symmetric.fn,
    },
    "tuned": {
        "pq": tuned.pq,
        "sq": tuned.sq,
        "rq": tuned.rq,
        "tp": tuned.tp,
        "fp": tuned.fp,
        "fn": tuned.fn,
    },
    "baseline": BASELINE,
    "chosen_settings": CHOSEN.values,
    "plateaus": {
        "threshold": threshold_plateau,
        "min_area": area_plateau,
        "join_gap": gap_plateau,
    },
    "calibration": {
        "monotonic": curve.is_monotonic,
        "middle_share": curve.middle_share,
        "middle_span": curve.middle_span,
        "spread": curve.spread,
        "mean_absolute_error": curve.mean_absolute_error,
        "baseline": BASELINE_CALIBRATION,
    },
    "best_epoch": result.best_epoch,
    "best_val_loss": result.best_val_loss,
    "training_minutes": sum(item.seconds for item in result.history) / 60,
}
(WORK / "phase4_summary.json").write_text(json.dumps(summary, indent=2))
first.table.to_csv(WORK / "sweep_threshold_area.csv", index=False)
second.table.to_csv(WORK / "sweep_join_gap.csv", index=False)
calibration_table.to_csv(WORK / "calibration.csv", index=False)

bundle = WORK / "phase4_votes_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for name in (
        "submission.csv",
        "submission_frozen.csv",
        "phase4_summary.json",
        "sweep_threshold_area.csv",
        "sweep_join_gap.csv",
        "calibration.csv",
        "calibration.png",
        "value_histogram.png",
        "overview.png",
        "contested.png",
    ):
        archive.write(WORK / name, arcname=name)
    for path in sorted(VAL_MAPS.glob("*.npy")):
        archive.write(path, arcname=f"prob_fold0_votes/{path.name}")

print(f"{bundle.name}: {bundle.stat().st_size / 1e6:.0f} MB")
print(json.dumps(summary, indent=2))

## 11. Reading the result

**Whether the change worked** is the symmetric comparison in section 5 -- the
same post-processing as the previous run -- and it takes `+0.01` PQ or better,
with an account of what moved it, before the change is worth keeping. The tuned
number further down is what the submission is worth, not what the change is
worth.

**Whether it did what it set out to do** is the calibration. If the middle of
the range is as empty as before, the threshold is still inert and the score
moved for some other reason.

The two can disagree, and that is informative rather than awkward. A flat
symmetric comparison with a clearly better tuned number says the model did not
improve but its output became something the post-processing can work with.